In [1]:
import json

import fitz
import json
import requests
import io
import pickle
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re
from bs4 import BeautifulSoup
import pandas as pd
import nltk
import random

In [2]:
# find pdf folder
source_dir = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/pdfs_only/"
len(os.listdir(source_dir))

100

In [ ]:
# for each PDF file:
# for each page in the PDF file
# collect info about its height and width
# store it

source_dir = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/pdfs_only/"
docs_pagesizes = {}
for filename in os.listdir(source_dir):
    try:
        #filename = [f for f in os.listdir(os.path.join(source_dir, dir)) if ".pdf" in f][0]
        filepath = os.path.join(source_dir, filename)
        doc = fitz.open(filepath)
        doc_pagesizes = []
        for n, p in enumerate(doc):
            pix = p.get_pixmap()
            page_data = {"page_index" : n, "page_width" : pix.width, "page_height" : pix.height}
            doc_pagesizes.append(page_data)
        docs_pagesizes[filename[:6]] = doc_pagesizes
    except:
        pass

In [ ]:
print("hello")

In [7]:
path_sanizited_textblocks = "../data/emlap_sanitized_textblocks"
textblocks_filenames = os.listdir(path_sanizited_textblocks)
textblocks_filenames[:10]

['100084_Croll1609_Basilica_chymica_MDZ_MBS.json',
 '100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json',
 '100058_Hagecius1596_Actio_medica_ER_UBB.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100069_Severinus1571_Idea_medicinae_philosophicae_GB_Noscemus.json',
 '100074_Hoghelande1595_De_lapidis_physici_conditionibus_MDZ_MBS.json',
 '100059_Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB.json',
 '100053_Mirandola1586_De_auro_libri_tres_MDZ_MBS.json',
 '100060_Witestein1583_Disceptatio_philosophica_MDZ_MBS.json',
 '100071_Pseudo-Lull1566_Testamentum_MDZ_MBS.json']

In [14]:
# test with individual document
filename = textblocks_filenames[0]
# load textblocks
with open(os.path.join(path_sanizited_textblocks, filename), 'r', ) as f:
    doc_textblocks_json = json.load(f)
# map on it corresponding previously extracted page sizes
doc_pagesizes = docs_pagesizes[filename[:6]]

In [23]:
doc_textblocks_recalculated = []

for  page_textblocks, page_size in zip(doc_textblocks_json, doc_pagesizes):
    page_width = page_size["page_width"]
    page_height = page_size["page_height"]
    page_textblocks_recalculated = []
    for textblock_data in page_textblocks:
                    textblock_data_recalculated = {
                        "coordinates":
                            {"upper_left_x": textblock_data["coordinates"][0] / page_width,
                             "upper_left_y": textblock_data["coordinates"][1] / page_height,
                             "lower_right_x" : textblock_data["coordinates"][2] / page_width,
                             "lower_right_y": textblock_data["coordinates"][3] / page_height},
                        "text": textblock_data["text"],
                        "tag": textblock_data["tag"]}
                    page_textblocks_recalculated.append(textblock_data_recalculated)
    doc_textblocks_recalculated.append(page_textblocks_recalculated)
with open(os.path.join(dir_textblocks, filename.replace(".json", "_recalculated.json")), 'w', ) as f:
    json.dump(doc_textblocks_recalculated, f, indent=2, ensure_ascii=False)



In [ ]:
with open(os.path.join(dir_textblocks, filename.replace(".json", "_recalculated.json")), 'w', ) as f:
    json.dump(doc_textblocks_recalculated, f, indent=2, ensure_ascii=False)

In [28]:
source_dir

'/srv/data/tome/tome-corpus/emlap_raw_2025-04-08/'

In [31]:
dir_textblocks = "../data/emlap_annotated_textblocks/"
for filename in os.listdir(dir_textblocks):
    if ".json" in filename and not "params" in filename and not "recalculated" in filename:
        with open(os.path.join(dir_textblocks, filename), 'r', ) as f:
            doc_textblocks_json = json.load(f)
        try:
            print("working on: ", filename, " ...")
            doc_pagesizes = docs_pagesizes[filename.replace(".json", ".pdf")]
            doc_textblocks_recalculated = []
            for page_textblocks, page_sizes in zip(doc_textblocks_json, doc_pagesizes):
                page_width = page_sizes["page_width"]
                page_height = page_sizes["page_height"]
                page_textblocks_recalculated = []
                for textblock_data in page_textblocks:
                    textblock_data_recalculated = {
                        "coordinates":
                            {"upper_left_x": textblock_data["coordinates"][0] / page_width,
                             "upper_left_y": textblock_data["coordinates"][1] / page_height,
                             "lower_right_x" : textblock_data["coordinates"][2] / page_width,
                             "lower_right_y": textblock_data["coordinates"][3] / page_height},
                        "text": textblock_data["text"],
                        "tag": textblock_data["tag"]}
                    page_textblocks_recalculated.append(textblock_data_recalculated)
                doc_textblocks_recalculated.append(page_textblocks_recalculated)
            with open(os.path.join(dir_textblocks, filename.replace(".json", "_recalculated.json")), 'w', encoding='utf-8') as f:
                json.dump(doc_textblocks_recalculated, f, indent=2, ensure_ascii=False)
        except:
            pass

working on:  Moffett_De_iure_et_praestantia_MDZ_MBS.json  ...
working on:  Toxites1567_Spongia_stibii_MDZ_MBS.json  ...
working on:  Pseudo-Lull1567_Mercuriorum_liber_MDZ_MBS.json  ...
working on:  Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json  ...
working on:  Pseudo-Aquinas1579_Secreta_alchemiae_magnalia_ONB.json  ...
working on:  Pseudo-Paracelsus1568_Pyrophilia_vexationumque_ONB.json  ...
working on:  Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.json  ...
working on:  Hagecius1585_De_cervisia_ejusque_conficiendi_ratione.json  ...
working on:  Pedemontanus1563_De_Secretis_MDZ_MBS.json  ...
working on:  Augurello,_Giovanni_Aurelio_-_Chrysopoeia__Venice_1515_pdf.json  ...
working on:  Phaedro1562_Aquila_coelestis_MBZ_MBS.json  ...
working on:  Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB.json  ...
working on:  Ulstadt1526_De_epidemia_ONB.json  ...
working on:  Gessner1569_Thesaurus_Euonymi_Philiatri_Liber_Secundus_MDZ_MBS.json  ...
working on:  Dorn1567_Clavis_totius_p